In [1]:
import torch
import torch.nn as nn
import torch.functional as F
import math


In [11]:

X_text = "He likes books".lower().split()
Y_text = "Usy books pasand hen".lower().split()

eng_vocab = {word: i for i, word in enumerate(set(X_text))}
urdu_vocab = {word: i for i, word in enumerate(set(Y_text))}

X_ids = torch.tensor([eng_vocab[w] for w in X_text])
Y_ids = torch.tensor([urdu_vocab[w] for w in Y_text])

d_model = 512 

encoder_embedding = nn.Embedding(num_embeddings=len(eng_vocab), embedding_dim=d_model)
decoder_embedding = nn.Embedding(num_embeddings=len(urdu_vocab), embedding_dim=d_model)

Xe = encoder_embedding(X_ids)
Ye = decoder_embedding(Y_ids)


In [2]:
class Positional_Embedding:
    def __init__(self, max_seq_len=100, input_dim=512):
        
        self.pe = torch.zeros(max_seq_len, input_dim, requires_grad=False)
        
        for pos in range(max_seq_len):
            for i in range(input_dim // 2):
            
                denominator = 10000 ** ((2 * i) / input_dim)
                ang = pos / denominator
                
                self.pe[pos, 2 * i] = math.sin(ang)
                
                self.pe[pos, 2 * i + 1] = math.cos(ang)

    def forward(self, X):
        seq_len = X.shape[1]
        return X + self.pe[:seq_len, :]

In [24]:
class Self_Attention_Head:
    def __init__(self, input_dim=512, head_dim=64):
        self.Wq = (torch.randn(input_dim, head_dim) * 0.01).requires_grad_()
        self.Wk = (torch.randn(input_dim, head_dim) * 0.01).requires_grad_()
        self.Wv = (torch.randn(input_dim, head_dim) * 0.01).requires_grad_()

    def forward(self, X):
        Q = X @ self.Wq
        K = X @ self.Wk
        V = X @ self.Wv
        
        K_T = K.transpose(-2, -1)
        z = Q @ K_T
        s = z / math.sqrt(K.shape[-1])
        p = torch.softmax(s, dim=-1)
        Ae = p @ V  
        return Ae

    def backward(self, lr):
        # Turn off gradient tracking during the actual update
        with torch.no_grad():
            
            # 1. Shift the weights down the gradient slope
            self.Wq -= lr * self.Wq.grad
            self.Wk -= lr * self.Wk.grad
            self.Wv -= lr * self.Wv.grad
            
            # 2. Wipe the gradients clean for the next batch
            self.Wq.grad.zero_()
            self.Wk.grad.zero_()
            self.Wv.grad.zero_()


class Multi_Head:
    def __init__(self, input_dim=512, num_heads=8):
        self.W = (torch.randn(input_dim, input_dim) * 0.01).requires_grad_()        
        self.b = torch.zeros(1, input_dim).requires_grad_()
        self.heads = []
        
        head_dim = input_dim // num_heads
        
        for i in range(num_heads):
            h = Self_Attention_Head(input_dim=input_dim, head_dim=head_dim)
            self.heads.append(h)
            
    def forward(self, X):
        head_outputs = []
        for h in self.heads:
            out = h.forward(X)
            head_outputs.append(out)
            
        concat_out = torch.cat(head_outputs, dim=-1)
        final_out = (concat_out @ self.W) + self.b
        return final_out

    def backward(self, lr):
        with torch.no_grad():
            # 1. Update the Multi-Head's own projection weights
            self.W -= lr * self.W.grad
            self.b -= lr * self.b.grad
            
            self.W.grad.zero_()
            self.b.grad.zero_()
            
        # 2. Command all 8 child heads to update their weights!
        for h in self.heads:
            h.backward(lr)

In [25]:
class Normalization:
    def __init__(self, input_dim=512, eps=1e-5):
        self.eps = eps 
        self.gamma = torch.ones(1, input_dim).requires_grad_()       
        self.beta = torch.zeros(1, input_dim).requires_grad_()   
    
    def forward(self, X):
        mean = X.mean(dim=-1, keepdim=True)
        var = X.var(dim=-1, unbiased=False, keepdim=True)
        
        X_normalized = (X - mean) / torch.sqrt(var + self.eps)
        out = (self.gamma * X_normalized) + self.beta
        
        return out

    def backward(self, lr):
        """ The LayerNorm Optimizer Step """
        with torch.no_grad():
            # 1. Update the learned scaling and shifting parameters
            self.gamma -= lr * self.gamma.grad
            self.beta -= lr * self.beta.grad
            
            # 2. Wipe the gradients clean
            self.gamma.grad.zero_()
            self.beta.grad.zero_()

In [26]:
class FFN:
    def __init__(self, input_dim=512, hidden_dim=1024):
        self.W1 = (torch.randn(input_dim, hidden_dim) * 0.01).requires_grad_()      
        self.b1 = torch.zeros(1, hidden_dim).requires_grad_()
        
        self.W2 = (torch.randn(hidden_dim, input_dim) * 0.01).requires_grad_()     
        self.b2 = torch.zeros(1, input_dim).requires_grad_()
    
    def forward(self, X):
        z1 = X @ self.W1 + self.b1
        l1 = torch.relu(z1)
        out = l1 @ self.W2 + self.b2
        
        return out

    def backward(self, lr):
        """ The FFN Optimizer Step """
        with torch.no_grad():
            # 1. Update the expansion and compression weights
            self.W1 -= lr * self.W1.grad
            self.b1 -= lr * self.b1.grad
            self.W2 -= lr * self.W2.grad
            self.b2 -= lr * self.b2.grad
            
            # 2. Wipe the gradients clean for the next batch
            self.W1.grad.zero_()
            self.b1.grad.zero_()
            self.W2.grad.zero_()
            self.b2.grad.zero_()

In [6]:

class Resedual_Connection:
    def __init__(self):
        pass
        
    def forward(self, current_out, prev_out):
        self.Re = current_out + prev_out
        return self.Re

In [27]:
class Mask_Attention_Head:
    def __init__(self, input_dim=512, head_dim=64):
        self.Wq = (torch.randn(input_dim, head_dim) * 0.01).requires_grad_()
        self.Wk = (torch.randn(input_dim, head_dim) * 0.01).requires_grad_()
        self.Wv = (torch.randn(input_dim, head_dim) * 0.01).requires_grad_()

    def forward(self, X):
        Q = X @ self.Wq
        K = X @ self.Wk
        V = X @ self.Wv
        
        K_T = K.transpose(-2, -1)
        z = Q @ K_T
        s = z / math.sqrt(K.shape[-1])
        
        seq_len = X.shape[1]
        tril_matrix = torch.tril(torch.ones(seq_len, seq_len))
        s_masked = s.masked_fill(tril_matrix == 0, float('-inf'))
        
        p = torch.softmax(s_masked, dim=-1)
        Ae = p @ V
        return Ae

    def backward(self, lr):
        """ The Masked Head Optimizer Step """
        with torch.no_grad():
            # Update the 3 projection matrices
            self.Wq -= lr * self.Wq.grad
            self.Wk -= lr * self.Wk.grad
            self.Wv -= lr * self.Wv.grad
            
            # Wipe gradients
            self.Wq.grad.zero_()
            self.Wk.grad.zero_()
            self.Wv.grad.zero_()


class Masked_Multi_Head:
    def __init__(self, input_dim=512, num_heads=8):
        self.W = (torch.randn(input_dim, input_dim) * 0.01).requires_grad_()        
        self.b = torch.zeros(1, input_dim).requires_grad_()
        self.heads = []
        
        head_dim = input_dim // num_heads
        
        for i in range(num_heads):
            h = Mask_Attention_Head(input_dim=input_dim, head_dim=head_dim)
            self.heads.append(h)
            
    def forward(self, X):
        head_outputs = []
        for h in self.heads:
            out = h.forward(X)
            head_outputs.append(out)
            
        concat_out = torch.cat(head_outputs, dim=-1)
        final_out = (concat_out @ self.W) + self.b
        return final_out

    def backward(self, lr):
        """ The Masked Multi-Head Optimizer Step """
        with torch.no_grad():
            # 1. Update its own final projection weights
            self.W -= lr * self.W.grad
            self.b -= lr * self.b.grad
            
            self.W.grad.zero_()
            self.b.grad.zero_()
            
        # 2. Command the 8 masked child heads to update!
        for h in self.heads:
            h.backward(lr)

In [28]:
class Cross_Attention_Head:
    def __init__(self, input_dim=512, head_dim=64):
        self.Wq = (torch.randn(input_dim, head_dim) * 0.01).requires_grad_()
        self.Wk = (torch.randn(input_dim, head_dim) * 0.01).requires_grad_()
        self.Wv = (torch.randn(input_dim, head_dim) * 0.01).requires_grad_()

    def forward(self, X_dec, Enc_out):
        Q = X_dec @ self.Wq
        K = Enc_out @ self.Wk
        V = Enc_out @ self.Wv
        
        K_T = K.transpose(-2, -1)
        z = Q @ K_T
        s = z / math.sqrt(K.shape[-1])
        
        p = torch.softmax(s, dim=-1)
        Ae = p @ V  
        return Ae

    def backward(self, lr):
        """ The Bridge Optimizer Step """
        with torch.no_grad():
            # 1. Update the translation bridge weights
            self.Wq -= lr * self.Wq.grad
            self.Wk -= lr * self.Wk.grad
            self.Wv -= lr * self.Wv.grad
            
            # 2. Wipe gradients clean
            self.Wq.grad.zero_()
            self.Wk.grad.zero_()
            self.Wv.grad.zero_()


class Cross_Multi_Head:
    def __init__(self, input_dim=512, num_heads=8):
        self.W = (torch.randn(input_dim, input_dim) * 0.01).requires_grad_()        
        self.b = torch.zeros(1, input_dim).requires_grad_()
        self.heads = []
        
        head_dim = input_dim // num_heads
        
        for i in range(num_heads):
            h = Cross_Attention_Head(input_dim=input_dim, head_dim=head_dim)
            self.heads.append(h)
            
    def forward(self, X_dec, Enc_out):
        head_outputs = []
        for h in self.heads:
            out = h.forward(X_dec, Enc_out)
            head_outputs.append(out)
            
        concat_out = torch.cat(head_outputs, dim=-1)
        final_out = (concat_out @ self.W) + self.b
        return final_out

    def backward(self, lr):
        """ The Cross Multi-Head Optimizer Step """
        with torch.no_grad():
            # 1. Update its own final projection weights
            self.W -= lr * self.W.grad
            self.b -= lr * self.b.grad
            
            self.W.grad.zero_()
            self.b.grad.zero_()
            
        # 2. Command the 8 cross-attention child heads to update
        for h in self.heads:
            h.backward(lr)

In [29]:
class Linear:
    def __init__(self, input_dim=512, vocab_size=50000):
        self.W = (torch.randn(input_dim, vocab_size) * 0.01).requires_grad_()       
        self.b = torch.zeros(1, vocab_size).requires_grad_()
          
    def forward(self, X):
        # 1. Project the 512-dim thought into the Vocabulary space
        logits = X @ self.W + self.b
        
        # 2. Apply Softmax to convert raw scores into percentages
        probabilities = torch.softmax(logits, dim=-1)
        
        return probabilities

    def backward(self, lr):
        """ The Final Output Projection Optimizer Step """
        with torch.no_grad():
            # 1. Update the massive vocabulary projection weights
            self.W -= lr * self.W.grad
            self.b -= lr * self.b.grad
            
            # 2. Wipe gradients clean
            self.W.grad.zero_()
            self.b.grad.zero_()

In [22]:
def calculate_loss(probabilities, target_labels):
    
    batch_size, seq_len, vocab_size = probabilities.shape
    
    # 1. Flatten the predictions and targets so we can compare them word-by-word
    probs_flat = probabilities.view(-1, vocab_size)
    targets_flat = target_labels.reshape(-1)
    
    # 2. Extract the probability the model gave to the EXACT correct word
    # Advanced indexing: We pluck the probability score of the true target word
    correct_word_probs = probs_flat[torch.arange(len(targets_flat)), targets_flat]
    
    # 3. Apply the Negative Log math (Adding a tiny epsilon to prevent log(0) crashes)
    loss = -torch.log(correct_word_probs + 1e-9)
    
    # 4. Return the average loss across all words in the batch
    return loss.mean()

In [30]:
class Encoder:
    def __init__(self, input_dim=512):
        self.multi_head = Multi_Head(input_dim)
        self.norm1 = Normalization(input_dim)
        self.res1 = Resedual_Connection()
        
        self.ffn = FFN(input_dim)
        self.norm2 = Normalization(input_dim)
        self.res2 = Resedual_Connection()

    def forward(self, X):
        attn_out = self.multi_head.forward(X)
        
        res1_out = self.res1.forward(current_out=attn_out, prev_out=X)
        
        norm1_out = self.norm1.forward(res1_out)
        
        ffn_out = self.ffn.forward(norm1_out)
        
        res2_out = self.res2.forward(current_out=ffn_out, prev_out=norm1_out)
        final_out = self.norm2.forward(res2_out)
        

        return final_out

    def backward(self, lr):
        """ The Encoder Command Routing """
        self.multi_head.backward(lr)
        self.norm1.backward(lr)
        
        self.ffn.backward(lr)
        self.norm2.backward(lr)


class Decoder:
    def __init__(self, input_dim=512):
        # 1. Masked Self-Attention
        self.mask_attention = Masked_Multi_Head(input_dim)
        self.norm1 = Normalization(input_dim)
        self.res1 = Resedual_Connection()
        
        # 2. Cross-Attention 
        self.cross_attention = Cross_Multi_Head(input_dim)
        self.norm2 = Normalization(input_dim)
        self.res2 = Resedual_Connection()
        
        # 3. Feed Forward Network
        self.ffn = FFN(input_dim)
        self.norm3 = Normalization(input_dim)
        self.res3 = Resedual_Connection()

    def forward(self, Prev_out, Enc_out):
        # --- Sub-Layer 1: Masked Self-Attention ---
        # The Decoder looks at its own past words (Prev_out)
        masked_attn_out = self.mask_attention.forward(Prev_out)
        
        # Add & Norm
        res1_out = self.res1.forward(current_out=masked_attn_out, prev_out=Prev_out)
        norm1_out = self.norm1.forward(res1_out)
        
        # --- Sub-Layer 2: Cross-Attention ---
        # The Decoder (norm1_out) asks the Encoder (Enc_out) for context
        cross_attn_out = self.cross_attention.forward(X_dec=norm1_out, Enc_out=Enc_out)
        
        # Add & Norm
        res2_out = self.res2.forward(current_out=cross_attn_out, prev_out=norm1_out)
        norm2_out = self.norm2.forward(res2_out)
        
        # --- Sub-Layer 3: Feed Forward Network ---
        # The Decoder processes the combined translation thought
        ffn_out = self.ffn.forward(norm2_out)
        
        # Add & Norm
        res3_out = self.res3.forward(current_out=ffn_out, prev_out=norm2_out)
        final_out = self.norm3.forward(res3_out)

        return final_out

    def backward(self, lr):
        """ The Decoder Command Routing """
        self.mask_attention.backward(lr)
        self.norm1.backward(lr)
        
        self.cross_attention.backward(lr)
        self.norm2.backward(lr)
        
        self.ffn.backward(lr)
        self.norm3.backward(lr)

In [59]:
class Transformer:
    def __init__(self, input_dim=512, eng_vocab_size=10000, urdu_vocab_size=12000):
        
        # 1. The Fuel Lines
        self.W_emb_enc = (torch.randn(eng_vocab_size, input_dim) * 0.01).requires_grad_()
        self.W_emb_dec = (torch.randn(urdu_vocab_size, input_dim) * 0.01).requires_grad_()
        self.pos_embedding = Positional_Embedding(max_seq_len=100, input_dim=input_dim)

        # 2. The 6 Encoder Layers
        self.enc1 = Encoder(input_dim)
        self.enc2 = Encoder(input_dim)
        self.enc3 = Encoder(input_dim)
        self.enc4 = Encoder(input_dim)
        self.enc5 = Encoder(input_dim)
        self.enc6 = Encoder(input_dim)
        
        # 3. The 6 Decoder Layers
        self.dec1 = Decoder(input_dim)
        self.dec2 = Decoder(input_dim)
        self.dec3 = Decoder(input_dim)
        self.dec4 = Decoder(input_dim)
        self.dec5 = Decoder(input_dim)
        self.dec6 = Decoder(input_dim)
        
        # 4. The Output Projection
        self.output_linear = Linear(input_dim=input_dim, vocab_size=urdu_vocab_size)

    def forward(self, X_enc_ids, X_dec_ids):
        """ THE TRAINING PASS: Everything happens at once in parallel. """

        # --- ENCODER PASS ---
        X_enc_emb = self.W_emb_enc[X_enc_ids]
        X_enc_emb = self.pos_embedding.forward(X_enc_emb)

        Enc1_out = self.enc1.forward(X_enc_emb)
        Enc2_out = self.enc2.forward(Enc1_out)
        Enc3_out = self.enc3.forward(Enc2_out)
        Enc4_out = self.enc4.forward(Enc3_out)
        Enc5_out = self.enc5.forward(Enc4_out)
        Enc6_out = self.enc6.forward(Enc5_out) # The Final Context Vector

        # --- DECODER PASS ---
        X_dec_emb = self.W_emb_dec[X_dec_ids]
        X_dec_emb = self.pos_embedding.forward(X_dec_emb)
        
        # Notice: Enc6_out is the bridge for EVERY decoder layer!
        Dec1_out = self.dec1.forward(X_dec_emb, Enc6_out)
        Dec2_out = self.dec2.forward(Dec1_out, Enc6_out)
        Dec3_out = self.dec3.forward(Dec2_out, Enc6_out)
        Dec4_out = self.dec4.forward(Dec3_out, Enc6_out)
        Dec5_out = self.dec5.forward(Dec4_out, Enc6_out)
        Dec6_out = self.dec6.forward(Dec5_out, Enc6_out)
        
        # --- FINAL PROJECTION ---
        logits = self.output_linear.forward(Dec6_out)
        return logits

    def inference(self, X_enc_ids, sos_token_id, eos_token_id, max_length=10):
        """ THE PREDICTION LOOP: Autoregressive generation. """
        
        # --- ENCODER PASS (Runs exactly ONCE) ---
        X_enc_emb = self.pos_embedding.forward(self.W_emb_enc[X_enc_ids])
        
        Enc1_out = self.enc1.forward(X_enc_emb)
        Enc2_out = self.enc2.forward(Enc1_out)
        Enc3_out = self.enc3.forward(Enc2_out)
        Enc4_out = self.enc4.forward(Enc3_out)
        Enc5_out = self.enc5.forward(Enc4_out)
        Enc6_out = self.enc6.forward(Enc5_out) # The Final Context Vector
        
        # --- DECODER LOOP ---
        current_y_ids = torch.tensor([[sos_token_id]])
        
        for step in range(max_length):
            # Embed the sequence we have generated so far
            Y_emb = self.W_emb_dec[current_y_ids]
            Y_emb = self.pos_embedding.forward(Y_emb)
            
            # Pass through all 6 Decoders
            Dec1_out = self.dec1.forward(Y_emb, Enc6_out)
            Dec2_out = self.dec2.forward(Dec1_out, Enc6_out)
            Dec3_out = self.dec3.forward(Dec2_out, Enc6_out)
            Dec4_out = self.dec4.forward(Dec3_out, Enc6_out)
            Dec5_out = self.dec5.forward(Dec4_out, Enc6_out)
            Dec6_out = self.dec6.forward(Dec5_out, Enc6_out)
            
            # Grab the output vector for the VERY LAST word generated
            last_token_vector = Dec6_out[:, -1, :] 
            
            # Predict the next word
            logits = self.output_linear.forward(last_token_vector)
            next_word_id = torch.argmax(logits, dim=-1).item()
            
            # Append to our sequence
            next_word_tensor = torch.tensor([[next_word_id]])
            current_y_ids = torch.cat([current_y_ids, next_word_tensor], dim=1)
            
            # Kill Switch
            if next_word_id == eos_token_id:
                break
                
        return current_y_ids

    def backward(self, lr):
        """ THE MASTER OPTIMIZER STEP """
        
        # 1. Update the Raw Word Embeddings (The Fuel Lines)
        with torch.no_grad():
            self.W_emb_enc -= lr * self.W_emb_enc.grad
            self.W_emb_dec -= lr * self.W_emb_dec.grad
            
            self.W_emb_enc.grad.zero_()
            self.W_emb_dec.grad.zero_()

        # 2. Command all 6 Encoders to update
        self.enc1.backward(lr)
        self.enc2.backward(lr)
        self.enc3.backward(lr)
        self.enc4.backward(lr)
        self.enc5.backward(lr)
        self.enc6.backward(lr)
        
        # 3. Command all 6 Decoders to update
        self.dec1.backward(lr)
        self.dec2.backward(lr)
        self.dec3.backward(lr)
        self.dec4.backward(lr)
        self.dec5.backward(lr)
        self.dec6.backward(lr)
        
        # 4. Command the Output Projection to update
        self.output_linear.backward(lr)

In [23]:
# A. Set up Vocabularies
eng_vocab = {"he": 0, "likes": 1, "books": 2}
urdu_vocab = {"<SOS>": 0, "<EOS>": 1, "usy": 2, "books": 3, "pasand": 4, "hen": 5}

# B. Create the Sequences
X_text = ["he", "likes", "books"]
X_ids = torch.tensor([[eng_vocab[w] for w in X_text]]) # Shape: (1, 3)

Y_text = ["<SOS>", "usy", "books", "pasand", "hen", "<EOS>"]
Y_ids = torch.tensor([[urdu_vocab[w] for w in Y_text]]) # Shape: (1, 6)


decoder_input = Y_ids[:, :-1]  

target_labels = Y_ids[:, 1:]   
# -------------------------------


model = Transformer(
    input_dim=512, 
    eng_vocab_size=len(eng_vocab), 
    urdu_vocab_size=len(urdu_vocab)
)


pred = model.forward(X_enc_ids=X_ids, X_dec_ids=decoder_input)

print(f"Shape of pred: {pred.shape}")
# Expected Output: torch.Size([1, 5, 6])
# (1 Batch, 5 Words Predicted, 6 Vocabulary Probabilities per word)

# E. Compute the Loss
loss = calculate_loss(probabilities=pred, target_labels=target_labels)

print(f"Initial Raw Loss: {loss.item():.4f}")

Shape of pred: torch.Size([1, 5, 6])
Initial Raw Loss: 1.8753


In [33]:
import torch

# 1. The 20-Sentence Training Dataset
dataset = [
    ("he likes books", "usy books pasand hen"),
    ("i am happy", "main khush hun"),
    ("she is reading", "wo parh rahi hai"),
    ("we play cricket", "hum cricket khailty hen"),
    ("they are eating", "wo kha rahy hen"),
    ("you look tired", "tum thakay lag rahy ho"),
    ("the sun is hot", "suraj garam hai"),
    ("sky is blue", "asman neela hai"),
    ("i love pakistan", "mujhay pakistan pasand hai"),
    ("ali goes to school", "ali school jata hai"),
    ("water is cold", "pani thanda hai"),
    ("he runs fast", "wo taiz bhagta hai"),
    ("dogs are barking", "kuttay bhonk rahy hen"),
    ("cat is sleeping", "billi so rahi hai"),
    ("my name is salim", "mera naam salim hai"),
    ("i am an engineer", "main aik engineer hun"),
    ("ai is the future", "ai mustaqbil hai"),
    ("deep learning is fun", "deep learning mazaydar hai"),
    ("i want to learn", "main seekhna chahta hun"),
    ("this is my paper", "yeh mera paper hai")
]

# 2. Build the Vocabularies Dynamically
eng_vocab = {"<UNK>": 0}
urdu_vocab = {"<SOS>": 0, "<EOS>": 1, "<UNK>": 2}

for eng, urdu in dataset:
    for word in eng.split():
        if word not in eng_vocab:
            eng_vocab[word] = len(eng_vocab)
    for word in urdu.split():
        if word not in urdu_vocab:
            urdu_vocab[word] = len(urdu_vocab)

print(f"English Vocab Size: {len(eng_vocab)}")
print(f"Urdu Vocab Size: {len(urdu_vocab)}")

English Vocab Size: 52
Urdu Vocab Size: 54


In [42]:
# 1. Initialize the custom Transformer
model = Transformer(
    input_dim=256,          # Scaled down to 128 so it runs fast on a CPU!
    eng_vocab_size=len(eng_vocab), 
    urdu_vocab_size=len(urdu_vocab)
)

epochs = 300
learning_rate = 0.05


for epoch in range(epochs):
    epoch_loss = 0
    
    for eng_sentence, urdu_sentence in dataset:
        
        x_tokens = [eng_vocab.get(w, 0) for w in eng_sentence.split()]
        X_ids = torch.tensor([x_tokens]) # Shape: (1, Seq_Len)
        
        # B. Tokenize Urdu and add <SOS> and <EOS>
        y_tokens = [urdu_vocab["<SOS>"]] + [urdu_vocab.get(w, 2) for w in urdu_sentence.split()] + [urdu_vocab["<EOS>"]]
        Y_ids = torch.tensor([y_tokens])
        
        
        decoder_input = Y_ids[:, :-1]  # <SOS> to last word
        target_labels = Y_ids[:, 1:]   # First word to <EOS>
        
        probabilities = model.forward(X_enc_ids=X_ids, X_dec_ids=decoder_input)
        
        loss = calculate_loss(probabilities, target_labels)
        epoch_loss += loss.item()
        
        # F. THE AUTOGRAD BACKWARD PASS
        loss.backward()
        
        # G. THE CUSTOM OPTIMIZER UPDATE
        model.backward(lr=learning_rate)
        
    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        avg_loss = epoch_loss / len(dataset)
        print(f"Epoch {epoch + 1}/{epochs} | Average Loss: {avg_loss:.4f}")



🚀 Starting Training...

Epoch 5/300 | Average Loss: 2.9506
Epoch 10/300 | Average Loss: 2.3250
Epoch 15/300 | Average Loss: 2.0822
Epoch 20/300 | Average Loss: 1.7613
Epoch 25/300 | Average Loss: 1.6571
Epoch 30/300 | Average Loss: 1.4320
Epoch 35/300 | Average Loss: 1.2327
Epoch 40/300 | Average Loss: 1.2068
Epoch 45/300 | Average Loss: 0.8587
Epoch 50/300 | Average Loss: 0.8691
Epoch 55/300 | Average Loss: 0.6780
Epoch 60/300 | Average Loss: 0.6261
Epoch 65/300 | Average Loss: 0.6568
Epoch 70/300 | Average Loss: 0.5824
Epoch 75/300 | Average Loss: 0.5565
Epoch 80/300 | Average Loss: 0.7050
Epoch 85/300 | Average Loss: 0.5637
Epoch 90/300 | Average Loss: 0.5931
Epoch 95/300 | Average Loss: 0.6025
Epoch 100/300 | Average Loss: 0.5924
Epoch 105/300 | Average Loss: 0.5804
Epoch 110/300 | Average Loss: 0.5698
Epoch 115/300 | Average Loss: 0.5498
Epoch 120/300 | Average Loss: 0.5355
Epoch 125/300 | Average Loss: 0.5259
Epoch 130/300 | Average Loss: 0.4952
Epoch 135/300 | Average Loss: 0.5

In [60]:

test_sentence = "ali goes to school"
print(f"English Input: {test_sentence}")

x_tokens = [eng_vocab.get(w, 0) for w in test_sentence.split()]
X_ids = torch.tensor([x_tokens])

generated_ids = model.inference(
    X_enc_ids=X_ids, 
    sos_token_id=urdu_vocab["<SOS>"], 
    eos_token_id=urdu_vocab["<EOS>"],
    max_length=20
)

# Convert IDs back to words
reverse_urdu_vocab = {v: k for k, v in urdu_vocab.items()}
translated_words = [reverse_urdu_vocab[id.item()] for id in generated_ids[0]]

print(f"Raw Output Tokens: {translated_words}")
clean_translation = " ".join([w for w in translated_words if w not in ["<SOS>", "<EOS>"]])
print(f"Final Urdu Translation: {clean_translation}")

English Input: ali goes to school
Step 0: Model generated token ID -> 29
Step 1: Model generated token ID -> 30
Step 2: Model generated token ID -> 31
Step 3: Model generated token ID -> 13
Step 4: Model generated token ID -> 1
Generated <EOS>. Stopping.
Raw Output Tokens: ['<SOS>', 'ali', 'school', 'jata', 'hai', '<EOS>']
Final Urdu Translation: ali school jata hai
